## 1. Loading the Cleaned Dataset

In this notebook, we will prepare the cleaned telecom customer dataset for machine learning.

The dataset was cleaned and saved during the data understanding and EDA stage. We will reuse that processed dataset instead of repeating the previous steps.

The goal of preprocessing is to transform the data into a suitable format for training machine learning models.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_telco_data.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Checking Duplicate Records

Duplicate records can affect machine learning results because the same observation may be counted more than once.

We will check whether the dataset contains any completely duplicated rows before proceeding with further preprocessing.

In [4]:
df.duplicated().sum()

np.int64(0)

## 3. Understanding Column Data Types

Machine learning algorithms generally require data in numerical format.

Our dataset contains both numerical and categorical features. We will identify their data types so that we can apply appropriate preprocessing techniques to each type.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 4. Identifying Numerical and Categorical Features

Numerical and categorical features require different preprocessing techniques.

- Numerical features may require scaling.
- Categorical features need to be converted into numerical representations using encoding techniques.

We will identify these features separately before applying preprocessing.

In [6]:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_cols = df.select_dtypes(include=["str"]).columns.tolist()

print("Numerical columns:", numerical_cols)
print("\nCategorical columns:", categorical_cols)

Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical columns: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


## 5. Separating Features and Target Variable

In supervised machine learning, we divide the dataset into two parts:

- **Features (X):** Input variables used by the model to make predictions.
- **Target (y):** The output variable that the model needs to predict.

In this project, the features contain customer information, while the target variable is `Churn`.

We will also exclude `customerID` because it is only an identifier and does not provide meaningful predictive information.

In [7]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7043, 19)
Target shape: (7043,)


## 6. Separating Numerical and Categorical Features

The input features contain both numerical and categorical data.

We will separate them because each type requires a different preprocessing approach.

- Numerical features: We will prepare them for scaling.
- Categorical features: We will convert their categories into numerical values using encoding.

The `SeniorCitizen` column will be treated as categorical because it represents two groups (0 and 1), rather than a continuous numerical measurement.

In [8]:
numerical_features = ["tenure", "MonthlyCharges", "TotalCharges"]

categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

categorical_features.append("SeniorCitizen")

print("Numerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

Numerical features: ['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']


## 7. Encoding Categorical Features

Categorical variables contain text values that machine learning algorithms cannot directly process.

We will use One-Hot Encoding to convert categorical values into numerical columns.

This technique creates a separate binary column for each category, representing its presence (1) or absence (0).

It is suitable for features such as Contract, PaymentMethod and InternetService, where categories do not have a natural numerical order.

## 8. Splitting the Dataset into Training and Testing Sets

We divide the dataset into training and testing sets before applying preprocessing.

- Training set: Used to train the machine learning model.
- Testing set: Used to evaluate the model on unseen data.

We use an 80:20 split, where 80% of the data is used for training and 20% for testing.

We also use stratified sampling to maintain approximately the same churn distribution in both sets.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (5634, 19)
Testing features: (1409, 19)
Training target: (5634,)
Testing target: (1409,)


## 9. Importing Preprocessing Tools

We will use Scikit-learn's preprocessing utilities to transform numerical and categorical features.

The following techniques will be used:

- `StandardScaler`: Standardizes numerical features so they have a mean of approximately 0 and a standard deviation of approximately 1.
- `OneHotEncoder`: Converts categorical values into numerical binary columns.
- `ColumnTransformer`: Applies different preprocessing operations to selected columns within the same workflow.

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

## 10. Standard Scaling of Numerical Features

Numerical features may have different ranges and scales.

We will use StandardScaler to standardize the numerical columns.

StandardScaler transforms the features so that they have a mean close to 0 and a standard deviation close to 1.

We will fit the scaler using only the training data to prevent data leakage. The same transformation will then be applied to the testing data.

In [11]:
scaler = StandardScaler()

X_train_numerical = scaler.fit_transform(X_train[numerical_features])

X_test_numerical = scaler.transform(X_test[numerical_features])

## 11. One-Hot Encoding of Categorical Features

Categorical features contain text-based values that need to be converted into numerical representations before training machine learning models.

We will use OneHotEncoder to create separate binary columns for each category.

We will use `handle_unknown="ignore"` so that the encoder can handle previously unseen categories in testing or future customer data without raising an error.

The encoder will be fitted only on the training data to prevent data leakage.

In [12]:
encoder = OneHotEncoder(handle_unknown="ignore")

X_train_categorical = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_categorical = encoder.transform(
    X_test[categorical_features]
)

## 12. Combining Numerical and Categorical Features

After scaling numerical features and encoding categorical features, we need to combine them into complete datasets.

We will use `hstack()` from SciPy to horizontally stack the transformed features.

This produces a single feature matrix containing both numerical and categorical information, ready for machine learning.

In [13]:
from scipy.sparse import hstack, csr_matrix

X_train_processed = hstack([
    csr_matrix(X_train_numerical),
    X_train_categorical
]).tocsr()

X_test_processed = hstack([
    csr_matrix(X_test_numerical),
    X_test_categorical
]).tocsr()

## 13. Inspecting the Processed Datasets

After applying scaling and encoding, the original 19 input features have been transformed into a larger set of numerical features.

One-Hot Encoding creates additional columns for individual categories.

We will inspect the dimensions of the processed training and testing datasets to ensure that both transformations were applied consistently.

In [14]:
print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)

print("\nTraining target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

Training data shape: (5634, 46)
Testing data shape: (1409, 46)

Training target shape: (5634,)
Testing target shape: (1409,)


## 14. Validating the Processed Data

Before using the processed data for machine learning, we need to verify its quality.

We will check for:
- Missing values (NaN).
- Infinite values.

This helps ensure that the transformed data is suitable for model training.

In [15]:
import numpy as np

print("Missing values:", np.isnan(X_train_processed.data).sum())

print("Infinite values:", np.isinf(X_train_processed.data).sum())

Missing values: 0
Infinite values: 0


## 15. Saving the Preprocessed Data

After completing the preprocessing operations, we will save the transformed training and testing datasets.

We will use SciPy's sparse matrix format to store the processed features efficiently.

This allows us to reuse the preprocessed data during model building and evaluation without repeating the preprocessing steps.

In [16]:
from scipy.sparse import save_npz

save_npz("../data/processed/X_train_processed.npz", X_train_processed)
save_npz("../data/processed/X_test_processed.npz", X_test_processed)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Preprocessed datasets saved successfully!")

Preprocessed datasets saved successfully!


In [17]:
import joblib

preprocessing_objects = {
    "scaler": scaler,
    "encoder": encoder,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features
}

joblib.dump(
    preprocessing_objects,
    "../models/preprocessing_objects.joblib"
)

print("Preprocessing objects saved successfully!")

Preprocessing objects saved successfully!
